In [ ]:
"""
    A practical example for fine-tuning an open-weight LLM using LoRA with Hugging Face Transformers and PEFT (Parameter-Efficient Fine-Tuning)

    We will use:
        - LLaMA-style workflow
        - LoRA (efficient fine-tuning)
        - Small instruction dataset
        - Single GPU setup
"""

In [ ]:
# Dependencies
# !pip install transformers datasets peft accelerate bitsandbytes trl ipywidgets hf_transfer
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, PeftModel
from datasets import load_dataset
from transformers import TrainingArguments                  # define the hyperparameters
from trl import SFTTrainer                                  # supervised fine-tuning trainer
from huggingface_hub.utils import enable_progress_bars

In [ ]:
# OS Pre-settings
os.environ["TOKENIZERS_PARALLELISM"] = "false"              # disable tokenizer multi-threading (parallelism)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"               # force fast Rust-based downloader

In [ ]:
# Load Model
enable_progress_bars()

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"         # 8-billion-parameter llama-3 model pre-quantized using BitsAndBytes into the 4-bit nf4 format

print(">>> Downloading/Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)       # download and initialize the tokenizer
tokenizer.pad_token = tokenizer.eos_token                   # set the pad token
print(">>> Tokenizer loaded successfully!")

print(">>> Downloading/Loading Model...")
model = AutoModelForCausalLM.from_pretrained(               # load the actual LLM into memory
    model_name,
    device_map="cuda:0",                                    # force the entire model onto GPU 0
    dtype=torch.bfloat16,                                   # switch from Float32 to bfloat16 -> halve the amount of VRAM
)
model = prepare_model_for_kbit_training(model)              # prepare the model for LoRA
print(">>> Model loaded successfully!")

In [ ]:
# Configure LoRA (Low-Rank Adaptation): LoRA trains only small adapter matrices instead of the full model
lora_config = LoraConfig(
    r=16,                                           # the size of the low-rank matrices injected into the model
    lora_alpha=32,                                  # scaling factor for the LoRA weights, normally set to double the value of r
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "gate_proj", 
        "up_proj", 
        "down_proj"
    ],                                              # for llama-3 model, target all linear modules
    lora_dropout=0.05,                              # randomly deactivate 5% of the LoRA parameters to avoid overfitting
    bias="none",                                    # bias parameters will not be trained
    task_type="CAUSAL_LM"                           # training a Causal Language Model
)

model = get_peft_model(model, lora_config)          # wrap the base model with the lora_config parameters

print(">>> Model training device:", next(model.parameters()).device)
model.print_trainable_parameters()

In [ ]:
# Format Dataset
dataset = load_dataset("json", data_files="train.json")

def format_example(example):
    # Structure the data as a conversation
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": example['input']},
        {"role": "assistant", "content": example['output']}
    ]
    
    # Apply the official Llama 3 template 
    # tokenize=False ensures it outputs text, which the map/trainer function expects
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    
    return {"text": text}

# Map the dataset
dataset = dataset.map(format_example)

In [ ]:
# Train
training_args = TrainingArguments(              # training hyperparameters
    output_dir="./results",                     # training checkpoint saving directory
    per_device_train_batch_size=2,              # number of training examples processed simultaneously on a single GPU
    gradient_accumulation_steps=3,              # accumulate the gradients for 3 steps before performing a weight update -> effective batch size = 6
    learning_rate=2e-4,                         # peak learning rate
    lr_scheduler_type="cosine",                 # smoothly decays the LR using a cosine curve
    warmup_ratio=0.03,                          # warm up for the first 3% of total steps
    num_train_epochs=12,                        # number of epochs
    logging_steps=1,                            # print training metrics every step
    save_strategy="epoch",                      # save a checkpoint automatically at the end of every epoch
    fp16=False,
    bf16=True
)

trainer = SFTTrainer(                           # initialize supervised fine-tuning trainer
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",  
    max_seq_length=512,         
    tokenizer=tokenizer,
    args=training_args                          # hyperparameter configuration settings
)

In [ ]:
# Start Training
trainer.train()

In [ ]:
# Save Model and Tokenizer
trainer.model.save_pretrained("fine_tuned_model")   # only save the tiny LoRA adapters instead of the massive file containing all 8 billion parameters
tokenizer.save_pretrained("fine_tuned_model")

In [ ]:
# Load Base and Fine-tuned Models
"""
    Since we only saved the adapters to keep things lightweight, we can't just load this folder by itself. 
    When we want to use the fine-tuned model, we need to load the base model first, then layer the saved adapters on top of it.
"""

# 1. Force the quantization config to map directly to bfloat16/float16 GPU execution
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# 2. Load base model directly to CUDA
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=quantization_config,
    device_map={"":"cuda:0"}
)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Load the adapter layer and forcefully bind it to the same GPU device
ft_model = PeftModel.from_pretrained(
    base_model,                         # base foundational model
    "fine_tuned_model"                  # path to the trained adapter weights
)
ft_tokenizer = AutoTokenizer.from_pretrained("fine_tuned_model")
ft_tokenizer.pad_token = ft_tokenizer.eos_token         # set the pad token

print(">>> Base Model Device:", base_model.device)
print(">>> Fine-Tuned Model Device:", ft_model.device)

In [ ]:
question = "How to optimize a Live Image?"
messages = [{"role": "user", "content": question}]
inputs = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True,                     # this stops at <|start_header_id|>assistant<|end_header_id|>
    return_tensors="pt"
).to("cuda")

input_length = inputs.shape[1]
outputs = ft_model.generate(inputs, max_new_tokens=200, eos_token_id=tokenizer.eos_token_id)
pure_answer_tokens = outputs[0][input_length:]
ft_answer = tokenizer.decode(pure_answer_tokens, skip_special_tokens=True).strip()

print(">>> QUESTION: ", question)
print(">>> FINE-TUNED LLM: ", ft_answer)